# Historial de penas acumulado

Dos calculos sobre la base:

1. Total de la cedula: la suma de las penas de todos los delitos de todos
   los procesos de esa persona.
2. Suma escalonada: suma la pena radicado por radicado en orden
   cronologico. Si el radicado 1 tiene 15 meses suma 15, y si el radicado
   2 tiene 30, la suma escalonada de ese radicado pasa a 45.

Salida: `base_historial_escalonado.parquet`.

In [ ]:
import re
from pathlib import Path

import numpy as np
import pandas as pd

DESCARGAS = Path.home() / "Downloads"

RUTA_BASE = DESCARGAS / "base_con_seguimiento.parquet"
RUTA_BASE_MUESTRA = DESCARGAS / "muestra_revision.csv"
RUTA_PENAS_CSV = DESCARGAS / "Penas_delitos_ajustado.csv"
RUTA_PENAS_XLSX = DESCARGAS / "Penas_delitos_ajustado.xlsx"
RUTA_SALIDA = DESCARGAS / "base_historial_escalonado.parquet"

COL_PROCESO = "radicado"
COL_ID = "IDENTIFICACION"
COL_FECHA = "FECHA"
COL_DELITO_SENTENCIA = "DelitoSentencia"
COL_DELITO_CAPTURA = "CONDUCTA"
COL_PENA = "pena_promedio_meses"

COL_LLAVE = "llave_caso"

RESPALDO_DELITO_CAPTURA = True

In [ ]:
if RUTA_BASE.exists():
    data = pd.read_parquet(RUTA_BASE)
    origen = RUTA_BASE.name
else:
    data = pd.read_csv(RUTA_BASE_MUESTRA, encoding="utf-8-sig")
    origen = RUTA_BASE_MUESTRA.name


print(f"Base cargada: {origen} ({len(data):,} filas)")

data[COL_FECHA] = pd.to_datetime(data[COL_FECHA], errors="coerce").astype("datetime64[ns]")
data[COL_PROCESO] = data[COL_PROCESO].astype("string")

data[COL_LLAVE] = (
    data[COL_ID].astype("string").fillna("SIN_CEDULA")
    + " | "
    + data[COL_PROCESO]
)

print(f"\nFilas (una por radicado + cédula + delito): {len(data):,}")
print(f"Capturas (radicado + cédula)              : {data[COL_LLAVE].nunique():,}")

In [ ]:
data["pena_delito"] = data[COL_PENA]
data["fuente_pena"] = np.where(data[COL_PENA].notna(), "delito sentenciado", None)

if RESPALDO_DELITO_CAPTURA:
    if RUTA_PENAS_CSV.exists():
        penas = pd.read_csv(RUTA_PENAS_CSV, encoding="utf-8-sig")
    elif RUTA_PENAS_XLSX.exists():
        penas = pd.read_excel(RUTA_PENAS_XLSX, sheet_name="penas")
    else:
        raise FileNotFoundError("No se encontró la tabla de penas por artículo.")

    penas["articulo"] = penas["articulo"].astype(str)
    pena_por_articulo = penas.set_index("articulo")[COL_PENA]

    articulo_captura = data[COL_DELITO_CAPTURA].str.extract(
        r"ART[IÍ]CULO\s+(\d+[A-Z]?)\.", flags=re.IGNORECASE
    )[0]

    pena_captura = articulo_captura.map(pena_por_articulo)

    primera_fila = ~data.duplicated(subset=[COL_LLAVE])
    usar_respaldo = data["pena_delito"].isna() & primera_fila & pena_captura.notna()

    data.loc[usar_respaldo, "pena_delito"] = pena_captura[usar_respaldo]
    data.loc[usar_respaldo, "fuente_pena"] = "delito de la captura"


data["fuente_pena"] = data["fuente_pena"].fillna("sin pena")

In [ ]:
por_captura = (
    data.groupby([COL_ID, COL_PROCESO, COL_LLAVE], dropna=False)
    .agg(
        fecha_captura=(COL_FECHA, "min"),
        penas_del_proceso=("pena_delito", "sum"),
        n_delitos=("pena_delito", "size")
    )
    .reset_index()
)

por_captura = por_captura.sort_values([COL_ID, "fecha_captura", COL_PROCESO])

por_captura["penas_acumuladas_radicado"] = (
    por_captura.groupby(COL_ID)["penas_del_proceso"].cumsum()
)

por_captura["delitos_acumulados_radicado"] = (
    por_captura.groupby(COL_ID)["n_delitos"].cumsum()
)

print(f"Capturas consolidadas: {len(por_captura):,}")

In [ ]:
totales = (
    por_captura.dropna(subset=[COL_ID])
    .groupby(COL_ID)
    .agg(
        penas_totales_cedula=("penas_del_proceso", "sum"),
        n_delitos_cedula=("n_delitos", "sum"),
        n_procesos_cedula=(COL_PROCESO, "nunique"),
    )
    .reset_index()
)
print(f"Cédulas analizadas: {len(totales):,}")

In [ ]:
data = data.merge(totales, on=COL_ID, how="left")

data = data.merge(
    por_captura[[COL_LLAVE, "penas_acumuladas_radicado", "delitos_acumulados_radicado"]],
    on=COL_LLAVE,
    how="left"
)

NUEVAS = [
    "penas_totales_cedula",
    "n_delitos_cedula",
    "n_procesos_cedula",
    "penas_acumuladas_radicado",
    "delitos_acumulados_radicado"
]

print("Cobertura de las variables nuevas (sobre la base larga):\n")
print((data[NUEVAS].notna().mean() * 100).round(1).to_string())

data.to_parquet(RUTA_SALIDA, index=False)

print(f"\nGuardado: {RUTA_SALIDA}")
print(f"{len(data):,} filas x {len(data.columns)} columnas")